# MCP Fundamentals

*Notebook 27*

Connect your agent to any MCP server and get its tools automatically — no custom integration code.
<br>
<br>

**Topics:**
- What MCP is and why it matters

- MCP servers, clients, and tools

- How MCP relates to @function_tool and built-in tools — three ways to give agents capabilities

- Connecting to a first MCP server with `MCPServerStdio`

- Tool discovery — the agent sees everything automatically

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

# Notebook-specific imports
import subprocess
import shutil

MODEL = "gpt-5-mini"

print("✅ Imports ready!")

Traces are automatically recorded and can be inspected in the OpenAI dashboard to see inputs, tool calls, and outputs together.

### Check Node.js / npx

The filesystem MCP server runs via `npx`. Verify Node.js is installed. Note: this is specific to the demo server — other MCP servers use other runtimes.

In [ ]:
# Check Node.js and npx availability
try:
    node_version = subprocess.check_output(["node", "--version"], text=True).strip()
    npx_version = subprocess.check_output(["npx", "--version"], text=True).strip()
    print(f"✅ Node.js: {node_version}")
    print(f"✅ npx: {npx_version}")
    print("\n✅ Ready for MCP!")
except FileNotFoundError:
    print("❌ Node.js not found")
    print("\nInstall Node.js from https://nodejs.org (LTS version)")
    print("After installing, restart JupyterLab and re-run this cell.")

---

## 🔌 Part 1: What Is MCP?

**MCP (Model Context Protocol)** is an open standard for connecting AI agents to external tools and data sources.

Think of it like USB-C for AI:
- Before USB-C, every device had its own connector — constant adapter chaos
- USB-C is one standard that works with everything
- Before MCP, every AI tool needed custom integration code
- MCP is one standard that works with any compliant agent or tool

**Three ways to give an agent capabilities:**

- `@function_tool` — you write the code; the tool runs locally in Python

- Built-in tools (e.g. web search) — OpenAI hosts and runs them; zero code needed

- MCP — a separate MCP server hosts the tool (running locally or remotely); your agent connects and uses it automatically. This is the third path: instant access to a growing ecosystem of community-maintained servers (GitHub, databases, Slack, calendars) with zero integration code.

MCP is the third path: no custom integration code, and any MCP-compatible agent works with any MCP server.

**The key insight:** an MCP server exposes tools in a standard format. Any MCP-compatible agent can connect to any MCP server and immediately use its tools — without knowing anything about how the server works internally.

### The Three Pieces

```
MCP Server   — exposes tools (filesystem, web, database, etc.)
MCP Client   — connects to the server (the Agents SDK is the client)
Agent        — uses the tools the server exposes
```

You write zero tool integration code. The server handles the implementation. The agent handles the reasoning. MCP handles the connection.

MCP servers can also expose **resources** — read-only data objects the agent can access directly, unlike tools which perform actions — though this notebook focuses on tools.

---

## 📁 Part 2: Your First MCP Server — Filesystem

The filesystem MCP server gives the agent tools to read, write, and navigate files. It runs as a local subprocess via `npx`.

We'll use `MCPServerStdio` — the SDK's client for local servers that communicate over standard input/output (stdio). Calling `list_tools()` yourself is how you inspect a new MCP server's surface before trusting an agent with it.

First, let's create a small workspace with sample files for the agent to explore.

In [ ]:
# Create a sample workspace for the agent
workspace = Path("mcp_workspace")
workspace.mkdir(exist_ok=True)

(workspace / "notes.txt").write_text("""Project Notes
==============
- MCP connects agents to external tools
- The filesystem server exposes read/write tools
- No custom integration code needed
- Works with any MCP-compatible agent
""")

(workspace / "tasks.txt").write_text("""TODO List
=========
1. Learn MCP fundamentals
2. Connect to filesystem server
3. Try web fetch server
4. Build MCP-powered assistant
""")

(workspace / "config.json").write_text('{"version": "1.0", "debug": false, "max_retries": 3}')

# --------------------------------------------------------------
print("✅ Workspace created")
print(f"    Location: {workspace.resolve()}")
print(f"    Files: {[f.name for f in workspace.iterdir()]}")

#### Connect and Discover Tools

In [ ]:
print("🔌 CONNECTING TO FILESYSTEM MCP SERVER")
print("=" * 60)

async with MCPServerStdio(
    name="Filesystem",
    params={
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(workspace.resolve())]
    },
    cache_tools_list=True
) as fs_server:

    # List available tools — the server tells us what it can do
    tools = await fs_server.list_tools()
    print(f"✅ Connected! {len(tools)} tools available:\n")
    for tool in tools:
        print(f"  🔧 {tool.name}")
        if hasattr(tool, 'description') and tool.description:
            print(f"     {tool.description[:80]}")

print("=" * 60)

⚠️ **Security note:** The filesystem MCP server may expose write and delete tools in addition to read tools. Always scope the server's directory access to only the path it needs — never point it at your home directory or project root. (See Lesson 23 for prompt injection risks with tool output.)

Note: MCP tools don't carry a `needs_approval` flag the way `@function_tool` does — Notebook 29 adds server-level approval for write and delete actions.

💡 **Tip:** Pass `cache_tools_list=True` to `MCPServerStdio` to fetch the tool list once and reuse it across calls — better performance when the server's tools don't change between runs.

---

## 🤖 Part 3: Agent with MCP Tools

Pass the MCP server to an agent via `mcp_servers=[server]`. The agent automatically sees and uses all the tools the server exposes — including write tools.

In [ ]:
print("🤖 AGENT WITH MCP FILESYSTEM TOOLS")
print("=" * 60)

async with MCPServerStdio(
    name="Filesystem",
    params={
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", str(workspace.resolve())]
    },
    cache_tools_list=True
) as fs_server:

    agent = Agent(
        name="FileAssistant",
        instructions=(
            "You are a file assistant with access to a workspace directory.\n"
            "Use the filesystem tools to read, list, and summarize files when asked."
        ),
        model=MODEL,
        mcp_servers=[fs_server]
    )

    # Ask the agent to explore the workspace
    result = await Runner.run(agent, input="List all files in the workspace and give me a brief summary of what each one contains.")
    print(result.final_output)

    # ---- Now let's write a file ----
    print("\n✍️ WRITE DEMO")
    print("-" * 60)

    result = await Runner.run(agent, input="Read tasks.txt and create a new file called summary.txt with a one-sentence summary of the task list.")
    print(result.final_output)

# Verify outside the MCP context — the file persists on disk
summary_path = workspace / "summary.txt"
if summary_path.exists():
    print(f"\n✅ summary.txt created:")
    print(summary_path.read_text())

print("=" * 60)

To use any other MCP server, you only change the `command` and `args` passed to `MCPServerStdio` — the `mcp_servers=[server]` line and the rest of the agent code stay exactly the same. The command and args for any server come from its documentation.

---

#### Cleanup

In [ ]:
shutil.rmtree(workspace, ignore_errors=True)
print("✅ Workspace cleaned up")

---

## 💪 Practice Exercises

### Exercise 1: File Organizer

*Covers: MCP filesystem server — file operations via MCP*

Create a workspace with several files and ask the agent to organize or summarize them.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: File Organizer
# --------------------------------------------------------------
# Objective: Use the filesystem MCP server to analyze a set of files.

# TODO 1: Create a workspace directory with at least 3 text files
#           containing different types of content (meeting notes,
#           a shopping list, a project plan, etc.)

# TODO 2: Connect to MCPServerStdio with the filesystem server
#           pointing at your workspace

# TODO 3: Create an agent with the MCP server

# TODO 4: Ask the agent to read all files and create a
#           consolidated index.txt listing each file with a
#           one-sentence description of its contents

# TODO 5: Verify index.txt was created and print its contents

# TODO 6: Clean up the workspace

# --- Write your code below this line ---

### Exercise 2: Discover Available Tools

*Covers: MCP introspection — listing available tools and descriptions*

Connect to the filesystem MCP server and print a formatted table of all available tools and their descriptions.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Tool Discovery
# --------------------------------------------------------------
# Objective: Explore what tools an MCP server exposes.

# TODO 1: Connect to MCPServerStdio (any directory is fine)

# TODO 2: Call server.list_tools() and store the result

# TODO 3: Print each tool's name and full description
#           formatted clearly (e.g., "Tool: read_file\n  Description: ...")

# TODO 4: Print the total count of available tools

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**MCP is the USB-C for AI tools:**
- One standard protocol — any MCP server works with any MCP client

- No custom integration code — the agent discovers tools automatically

- A growing ecosystem of pre-built MCP servers for common services
<br>
<br>

**Transport:**
- `MCPServerStdio` — local subprocess via `npx` or similar command
<br>
<br>

**Key patterns:**
- Always use `async with MCPServerStdio(...) as server:` — manages the connection lifecycle

- Pass `mcp_servers=[server]` to the agent — tools appear automatically

- Use `cache_tools_list=True` for better performance on stable servers

---

## 📍 Next Step

**Notebook 28: Real-World MCP Servers**  

Connect to the filesystem and web fetch MCP servers and combine multiple MCP servers in one agent.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-27-mcp-fundamentals)

---